## Send a message to Confluent-kafka Topic via Python

In [1]:
!pip install confluent-Kafka

In [3]:
from confluent_kafka import Producer
import json
import time

In [4]:
import os
os.listdir()

['opt',
 'boot',
 'mnt',
 'lib',
 'root',
 '.ipynb_checkpoints',
 'home',
 'dev',
 'var',
 'hadoop',
 'tmp',
 'media',
 'run',
 'sys',
 'dataforHDFS',
 'copyright',
 'proc',
 'lib64',
 'customers.json',
 'dates_data.csv',
 'lost+found',
 'bin',
 'srv',
 'usr',
 '.cache',
 'sbin',
 'etc']

In [4]:
import os

os.listdir('/dataforHDFS')

['.ipynb_checkpoints',
 'customers_10mb.csv',
 'customers_300mb.csv',
 'first_100_customers.csv',
 'customers_150mb.csv',
 'orders.csv',
 'customers.csv']

### We can process source data files such as csv, parquet, by transforming them to JSON format
1. Consider customers_10mb.csv file and convert the data to the JSON format, which are then published as messages to a Kafka topic.

In [5]:
### Getting the data file,to load into the topic

import pandas as pd

csv_file = '/dataforHDFS/first_100_customers.csv'

df = pd.read_csv(csv_file)
df.head()

,customer_id,name,city,state,country,registration_date,is_active
0,0,Customer_0,Pune,Maharashtra,India,2023-06-29,False
1,1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True
2,2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True
3,3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False
4,4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False


In [6]:
df.count()

customer_id          99
name                 99
city                 99
state                99
country              99
registration_date    99
is_active            99
dtype: int64

### Creating a JSON data format from csv format

In [7]:
## converts Pandas DataFrame (df) to list of dictionaries: Each row becomes one dictionary. Each column name becomes a key. Each cell value becomes the value

json_records = df.to_dict(orient = 'records') 

In [8]:
json_file = 'customers.json'

with open(json_file, 'w') as file:
    json.dump(json_records, file, indent = 4)
    
print("File converted to JSON Format")

File converted to JSON Format


In [12]:
json_records[:3]

[{'customer_id': 0,
  'name': 'Customer_0',
  'city': 'Pune',
  'state': 'Maharashtra',
  'country': 'India',
  'registration_date': '2023-06-29',
  'is_active': False},
 {'customer_id': 1,
  'name': 'Customer_1',
  'city': 'Bangalore',
  'state': 'Tamil Nadu',
  'country': 'India',
  'registration_date': '2023-12-07',
  'is_active': True},
 {'customer_id': 2,
  'name': 'Customer_2',
  'city': 'Hyderabad',
  'state': 'Gujarat',
  'country': 'India',
  'registration_date': '2023-10-27',
  'is_active': True}]

In [ ]:
## creating a Python dictionary that holds all the connection settings required for kafka: producer and consumer, and admin

configuration_dict = {

# Required connection configs for Kafka producer, consumer, and admin
"bootstrap.servers" : "x",  # Kafka broker address
"security.protocol" : "y", ## communication is secured: SASL_SSL means: SSL → encrypts data (secure transmission) and SASL → handles authentication
"sasl.mechanisms" : "PLAIN" , # PLAIN = username + password authentication
"sasl.username" : "a" ,  # API KEY and Secret
"sasl.password" : "b",

"session.timeout.ms" : 45000,
"client.id" : "z" # unique name for our application

}

producer = Producer(configuration_dict)


%4|1777337675.407|CONFWARN|ccloud-python-client-256b7da7-0dbb-426a-8107-7f5c7a209ba5#producer-1| [thrd:app]: Configuration property session.timeout.ms is a consumer property and will be ignored by this producer instance
%6|1777337676.656|GETSUBSCRIPTIONS|ccloud-python-client-256b7da7-0dbb-426a-8107-7f5c7a209ba5#producer-1| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to DABoXRbqQW6rOvKSmeDGFw


### Sending a single message to kafka topic 'ecommerce'

In [10]:
topic = 'ecommerce'

with open('customers.json', 'r') as file:
    customers_data = json.load(file) # loading the file into customers_data
    
value = customers_data[0]
key = value['customer_id']
print(key,value)

0 {'customer_id': 0, 'name': 'Customer_0', 'city': 'Pune', 'state': 'Maharashtra', 'country': 'India', 'registration_date': '2023-06-29', 'is_active': False}


In [17]:
type(value)

dict

In [11]:
# this is in the dict, to send data we need to change to byte

producer.produce(topic, key = str(key).encode('utf-8'), value = str(value).encode('utf-8') )

### Send multiple messages to kafka topic 'ecommerce'

In [12]:
def delivery_status(err, msg): # This function is a callback function. Kafka calls this function after trying to deliver each message.

    if err:
        # If there is an error while sending the message, err will contain the error details.
        print(f"Message Delivery failed: {err}")

    else:
        # If message delivery is successful, msg contains metadata about the delivered message.
        print(f"message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}") # Getting the topic name -> message was sent, partition number -> message was stored, position of  message inside that partition


for record in customers_data:
    # Loop through each record in customers_data.

    try:
        message_value = json.dumps(record) # Kafka cannot directly send a Python dictionary, so we convert it to JSON.

        message_key = str(int(record['customer_id']) + 1000).encode('utf-8') #Gets customer_id from record. Converts it to string. Encodes it to bytes using UTF-8. Key helps Kafka decide which partition the message should go to.

        producer.produce( topic, key = message_key, value = message_value, callback = delivery_status) # callback -> function called after Kafka confirms delivery or failure

        # producer.produce() is asynchronous.That means it does not wait until Kafka confirms delivery. It puts the message into the producer queue and moves to the next record.

        producer.poll(1)
        # poll() triggers Kafka delivery callbacks.
        # Without poll(), delivery_status may not execute immediately.
        # 1 means it waits up to 1 second for delivery events.

    except Exception as e:
        # If any error happens inside the try block, this catches it.
        print(f"Error Sending message: {e}")


producer.flush()
# Waits until all queued messages are delivered to Kafka.
# This is important because produce() is asynchronous.
# Without flush(), some messages may remain in the local producer queue.


print("Message sent to kafka successfully") # Prints final success message after flush completes.

message delivered to ecommerce [0] at offset 44
message delivered to ecommerce [2] at offset 25
message delivered to ecommerce [0] at offset 45
message delivered to ecommerce [2] at offset 26
message delivered to ecommerce [1] at offset 39
message delivered to ecommerce [2] at offset 27
message delivered to ecommerce [0] at offset 46
message delivered to ecommerce [1] at offset 40
message delivered to ecommerce [2] at offset 28
message delivered to ecommerce [2] at offset 29
message delivered to ecommerce [0] at offset 47
message delivered to ecommerce [1] at offset 41
message delivered to ecommerce [2] at offset 30
message delivered to ecommerce [2] at offset 31
message delivered to ecommerce [2] at offset 32
message delivered to ecommerce [2] at offset 33
message delivered to ecommerce [2] at offset 34
message delivered to ecommerce [1] at offset 42
message delivered to ecommerce [1] at offset 43
message delivered to ecommerce [0] at offset 48
message delivered to ecommerce [1] at of